# Workshop 06 — Jail System and Game Polish

The game from Workshop 05 already handles jail at a basic level, but in this workshop we will examine the jail system in detail, handle edge cases, and add polish such as a winner banner and better logging. This brings the game much closer to the finished product.

**What you will learn:**
- How the jail system works (doubles to escape, 3-turn limit, £50 fine)
- Defensive coding to handle edge cases
- Adding a winner banner to the UI
- Improving the game console with coloured, categorised messages

**Time:** roughly 15 minutes.

## Section 1 — How Jail Works

The jail logic happens at the *start* of a turn, before movement:

1. Player rolls dice.
2. If they are in jail:
   - **Rolled doubles?** They escape and move normally.
   - **Third turn in jail?** They pay a £50 fine and get out.
   - **Otherwise?** They stay in jail. Turn ends immediately.
3. If they are not in jail, they move normally.

This means `jailTurns` counts how many rolls the player has made while in jail.

In [ ]:
import os

project_folder = os.path.join(os.path.expanduser("~"), "text_monopoly")
os.makedirs(project_folder, exist_ok=True)

# Let us look at the jail logic in isolation
print("""Jail logic (runs inside rollDice, before movement):

if (p.inJail) {
    p.jailTurns++;

    if (d1 === d2) {
        // Doubles! Player escapes jail
        p.inJail = false;
        // Continue to normal movement below...

    } else if (p.jailTurns >= 3) {
        // Three turns served, forced to pay fine
        p.inJail = false;
        p.cash -= 50;
        // Continue to normal movement below...

    } else {
        // Still stuck in jail
        state.phase = "endturn";
        updateUI();
        return;  // IMPORTANT: return early, skip movement
    }
}
// ...normal movement happens here only if not stuck""")

## Section 2 — Adding the Winner Banner

When a player goes bankrupt, we should show a clear visual indicator. The `checkBankrupt` function already detects this. Now let us add a banner element to the DOM dynamically.

In [ ]:
print("""Winner banner code (added to checkBankrupt):

function checkBankrupt(p) {
    if (p.cash <= 0) {
        state.gameOver = true;
        const winner = state.players.find(pl => pl !== p);
        log(p.name + " is BANKRUPT!", "danger");
        log(winner.name + " WINS with £" + winner.cash + "!", "reward");

        // Disable all buttons
        document.getElementById("rollBtn").disabled = true;

        // Create and display a winner banner
        const banner = document.createElement("div");
        banner.className = "winner-banner";
        banner.textContent = winner.name + " Wins!";
        document.querySelector(".sidebar").appendChild(banner);
    }
}

CSS for the banner:
.winner-banner {
    text-align: center;
    padding: 16px;
    border-radius: 12px;
    font-size: 1.2rem;
    font-weight: bold;
    background: rgba(210, 153, 34, 0.2);
    border: 1px solid #d29922;
    color: #d29922;
    margin-top: 8px;
}""")

## Section 3 — The Polished Game

The cell below writes the near-complete game. It includes all the features from previous workshops plus the winner banner, improved jail display, and better log formatting.

In [ ]:
html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Text Monopoly - Polished</title>
    <style>
        :root {
            --bg: #0d1117; --surface: #161b22; --border: #30363d;
            --text: #e6edf3; --dim: #8b949e; --green: #3fb950;
            --blue: #58a6ff; --red: #f85149; --yellow: #d29922; --purple: #bc8cff;
        }
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { background: var(--bg); color: var(--text); font-family: Georgia, serif; padding: 20px; }
        h1 { color: var(--green); margin-bottom: 4px; font-size: 1.6rem; }
        .subtitle { color: var(--dim); font-size: 0.85rem; margin-bottom: 20px; }
        .game-container { display: grid; grid-template-columns: 1fr 320px; gap: 20px; max-width: 1100px; }
        .console { background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 16px; height: 520px; overflow-y: auto; font-size: 0.82rem; line-height: 1.7; }
        .console::-webkit-scrollbar { width: 6px; }
        .console::-webkit-scrollbar-thumb { background: var(--border); border-radius: 3px; }
        .sidebar { display: flex; flex-direction: column; gap: 12px; }
        .card { background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 16px; }
        .card h3 { margin-bottom: 8px; display: flex; align-items: center; gap: 8px; }
        .dot { width: 10px; height: 10px; border-radius: 50%; display: inline-block; }
        .dot-p1 { background: var(--green); }
        .dot-p2 { background: var(--blue); }
        .stat { display: flex; justify-content: space-between; padding: 3px 0; font-size: 0.85rem; }
        .label { color: var(--dim); }
        .value { font-weight: bold; }
        .props { margin-top: 8px; font-size: 0.75rem; }
        .props span { display: inline-block; background: var(--bg); border: 1px solid var(--border); border-radius: 4px; padding: 2px 8px; margin: 2px; color: var(--text); }
        .dice-display { text-align: center; font-size: 2.5rem; padding: 8px; letter-spacing: 12px; }
        .turn-indicator { text-align: center; font-size: 0.85rem; padding: 8px; border-radius: 8px; font-weight: bold; }
        .turn-p1 { background: rgba(63, 185, 80, 0.15); color: var(--green); }
        .turn-p2 { background: rgba(88, 166, 255, 0.15); color: var(--blue); }
        button { width: 100%; padding: 12px; border: 1px solid var(--border); border-radius: 8px; background: var(--bg); color: var(--text); font-size: 0.9rem; cursor: pointer; margin-bottom: 8px; font-weight: bold; transition: all 0.2s; }
        button:hover:not(:disabled) { border-color: var(--green); background: rgba(63,185,80,0.1); }
        button:disabled { opacity: 0.3; cursor: not-allowed; }
        .roll-btn { background: linear-gradient(135deg, var(--green), #238636); border: none; color: #fff; font-size: 1rem; padding: 14px; }
        .roll-btn:hover:not(:disabled) { background: linear-gradient(135deg, #46d160, #2ea043); }
        .log-system { color: var(--dim); } .log-player1 { color: var(--green); }
        .log-player2 { color: var(--blue); } .log-event { color: var(--yellow); }
        .log-danger { color: var(--red); } .log-reward { color: var(--purple); }
        .winner-banner { text-align: center; padding: 16px; border-radius: 12px; font-size: 1.2rem; font-weight: bold; background: rgba(210,153,34,0.2); border: 1px solid var(--yellow); color: var(--yellow); margin-top: 8px; }
    </style>
</head>
<body>
    <h1>🎲 Text Monopoly</h1>
    <p class="subtitle">Workshop 06 - jail system and game polish</p>
    <div class="game-container">
        <div class="console" id="console"></div>
        <div class="sidebar">
            <div class="turn-indicator" id="turnIndicator">Player 1's Turn</div>
            <div class="dice-display" id="diceDisplay">🎲 🎲</div>
            <div class="card" id="p1card"></div>
            <div class="card" id="p2card"></div>
            <div class="card">
                <button class="roll-btn" id="rollBtn" onclick="rollDice()">🎲 Roll Dice</button>
                <button id="buyBtn" onclick="buyProperty()" disabled>Buy Property</button>
                <button id="endTurnBtn" onclick="endTurn()" disabled>End Turn</button>
            </div>
        </div>
    </div>
    <script>
    // This is the same game code as Workshop 05 but with:
    // 1. Winner banner in checkBankrupt
    // 2. Improved jail display in updateUI
    // 3. Scrollbar styling on the console

    const BOARD=[{name:"GO",type:"go"},{name:"Old Kent Road",type:"property",cost:60,rent:10,color:"#8B4513"},{name:"Community Chest",type:"chest"},{name:"Whitechapel Rd",type:"property",cost:60,rent:10,color:"#8B4513"},{name:"Income Tax",type:"tax",amount:200},{name:"Kings Cross",type:"property",cost:200,rent:50,color:"#333"},{name:"Angel Islington",type:"property",cost:100,rent:20,color:"#87CEEB"},{name:"Chance",type:"chance"},{name:"Euston Road",type:"property",cost:100,rent:20,color:"#87CEEB"},{name:"Pentonville Rd",type:"property",cost:120,rent:25,color:"#87CEEB"},{name:"Just Visiting",type:"visiting"},{name:"Pall Mall",type:"property",cost:140,rent:30,color:"#FF00FF"},{name:"Electric Co.",type:"property",cost:150,rent:35,color:"#FFD700"},{name:"Whitehall",type:"property",cost:140,rent:30,color:"#FF00FF"},{name:"Northumberland",type:"property",cost:160,rent:35,color:"#FF00FF"},{name:"Marylebone Stn",type:"property",cost:200,rent:50,color:"#333"},{name:"Bow Street",type:"property",cost:180,rent:40,color:"#FF8C00"},{name:"Community Chest",type:"chest"},{name:"Marlborough St",type:"property",cost:180,rent:40,color:"#FF8C00"},{name:"Vine Street",type:"property",cost:200,rent:45,color:"#FF8C00"},{name:"Free Parking",type:"free"},{name:"Strand",type:"property",cost:220,rent:50,color:"#FF0000"},{name:"Chance",type:"chance"},{name:"Fleet Street",type:"property",cost:220,rent:50,color:"#FF0000"},{name:"Trafalgar Sq",type:"property",cost:240,rent:55,color:"#FF0000"},{name:"Fenchurch Stn",type:"property",cost:200,rent:50,color:"#333"},{name:"Leicester Sq",type:"property",cost:260,rent:60,color:"#FFFF00"},{name:"Coventry Street",type:"property",cost:260,rent:60,color:"#FFFF00"},{name:"Water Works",type:"property",cost:150,rent:35,color:"#FFD700"},{name:"Piccadilly",type:"property",cost:280,rent:65,color:"#FFFF00"},{name:"GO TO JAIL",type:"gotojail"},{name:"Regent Street",type:"property",cost:300,rent:70,color:"#006400"},{name:"Oxford Street",type:"property",cost:300,rent:70,color:"#006400"},{name:"Community Chest",type:"chest"},{name:"Bond Street",type:"property",cost:320,rent:75,color:"#006400"},{name:"Liverpool Stn",type:"property",cost:200,rent:50,color:"#333"},{name:"Chance",type:"chance"},{name:"Park Lane",type:"property",cost:350,rent:85,color:"#00008B"},{name:"Super Tax",type:"tax",amount:100},{name:"Mayfair",type:"property",cost:400,rent:100,color:"#00008B"}];

    const CHANCE_CARDS=[{text:"Advance to GO! Collect £200",action:function(p){p.position=0;p.cash+=200;}},{text:"Bank pays you dividend of £50",action:function(p){p.cash+=50;}},{text:"Go back 3 spaces",action:function(p){p.position=(p.position-3+40)%40;}},{text:"Go directly to Jail!",action:function(p){p.position=10;p.inJail=true;p.jailTurns=0;}},{text:"Building repairs: pay £150",action:function(p){p.cash-=150;}},{text:"You won a crossword! Collect £100",action:function(p){p.cash+=100;}},{text:"Speeding fine: pay £50",action:function(p){p.cash-=50;}}];

    const CHEST_CARDS=[{text:"Bank error in your favour! Collect £200",action:function(p){p.cash+=200;}},{text:"Doctor's fees: pay £50",action:function(p){p.cash-=50;}},{text:"Sale of stock: collect £45",action:function(p){p.cash+=45;}},{text:"Holiday fund matures: collect £100",action:function(p){p.cash+=100;}},{text:"Pay school fees of £150",action:function(p){p.cash-=150;}},{text:"Income tax refund: collect £20",action:function(p){p.cash+=20;}},{text:"Go to Jail! Do not pass GO",action:function(p){p.position=10;p.inJail=true;p.jailTurns=0;}}];

    let state={players:[{name:"Player 1",cash:1500,position:0,properties:[],inJail:false,jailTurns:0},{name:"Player 2",cash:1500,position:0,properties:[],inJail:false,jailTurns:0}],currentPlayer:0,dice:[0,0],phase:"roll",gameOver:false,turnCount:0};

    function log(t,c){var el=document.getElementById("console");var li=document.createElement("div");li.className="log-"+(c||"system");li.textContent="> "+t;el.appendChild(li);el.scrollTop=el.scrollHeight;}
    function getPropertyOwner(pos){for(var i=0;i<state.players.length;i++){if(state.players[i].properties.includes(pos))return i;}return null;}

    function checkBankrupt(p){
        if(p.cash<=0){
            state.gameOver=true;
            var w=state.players.find(function(pl){return pl!==p;});
            log(p.name+" is BANKRUPT!","danger");
            log(w.name+" WINS with £"+w.cash+"!","reward");
            document.getElementById("rollBtn").disabled=true;
            var banner=document.createElement("div");
            banner.className="winner-banner";
            banner.textContent="🏆 "+w.name+" Wins!";
            document.querySelector(".sidebar").appendChild(banner);
        }
    }

    function handleSpace(p,space,pClass){
        switch(space.type){
            case "go":p.cash+=200;log("Landed on GO! Collect £200","reward");state.phase="endturn";break;
            case "property":
                var owner=getPropertyOwner(p.position);
                if(owner===null){if(p.cash>=space.cost){state.phase="buy";log(space.name+" is available for £"+space.cost+". Buy it?","event");}else{log("Cannot afford "+space.name,"system");state.phase="endturn";}}
                else if(owner!==state.currentPlayer){p.cash-=space.rent;state.players[owner].cash+=space.rent;log("Owned by "+state.players[owner].name+"! Pay £"+space.rent+" rent","danger");state.phase="endturn";checkBankrupt(p);}
                else{log("You own this property","system");state.phase="endturn";}
                break;
            case "tax":p.cash-=space.amount;log("Tax! Pay £"+space.amount,"danger");state.phase="endturn";checkBankrupt(p);break;
            case "gotojail":p.position=10;p.inJail=true;p.jailTurns=0;log("GO TO JAIL! 🚔","danger");state.phase="endturn";break;
            case "chance":var c=CHANCE_CARDS[Math.floor(Math.random()*CHANCE_CARDS.length)];log("Chance: "+c.text,"event");c.action(p);state.phase="endturn";checkBankrupt(p);break;
            case "chest":var ch=CHEST_CARDS[Math.floor(Math.random()*CHEST_CARDS.length)];log("Community Chest: "+ch.text,"event");ch.action(p);state.phase="endturn";checkBankrupt(p);break;
            default:state.phase="endturn";break;
        }
    }

    function rollDice(){
        if(state.phase!=="roll"||state.gameOver)return;
        var d1=Math.floor(Math.random()*6)+1,d2=Math.floor(Math.random()*6)+1;
        state.dice=[d1,d2];var total=d1+d2;
        var de=['','⚀','⚁','⚂','⚃','⚄','⚅'];
        document.getElementById("diceDisplay").textContent=de[d1]+" "+de[d2];
        var p=state.players[state.currentPlayer];
        var pClass=state.currentPlayer===0?"player1":"player2";
        log(p.name+" rolled "+d1+" + "+d2+" = "+total,pClass);

        if(p.inJail){p.jailTurns++;if(d1===d2){p.inJail=false;log("Doubles! Escaped jail!","event");}else if(p.jailTurns>=3){p.inJail=false;p.cash-=50;log("Paid £50 fine to leave jail","danger");}else{log("Still in jail ("+p.jailTurns+"/3)","danger");state.phase="endturn";updateUI();return;}}

        var oldPos=p.position;p.position=(p.position+total)%40;
        if(p.position<oldPos&&p.position!==0){p.cash+=200;log("Passed GO! Collect £200","reward");}
        log("Landed on: "+BOARD[p.position].name,pClass);
        handleSpace(p,BOARD[p.position],pClass);updateUI();
    }

    function buyProperty(){if(state.phase!=="buy")return;var p=state.players[state.currentPlayer];var s=BOARD[p.position];p.cash-=s.cost;p.properties.push(p.position);log(p.name+" bought "+s.name+" for £"+s.cost+"!","reward");state.phase="endturn";updateUI();}
    function endTurn(){if(state.phase!=="buy"&&state.phase!=="endturn")return;state.currentPlayer=(state.currentPlayer+1)%2;state.phase="roll";state.turnCount++;updateUI();}

    function updateUI(){
        var cp=state.currentPlayer;
        var ti=document.getElementById("turnIndicator");
        ti.textContent=state.players[cp].name+"'s Turn";
        ti.className="turn-indicator turn-p"+(cp+1);
        var dots=["dot-p1","dot-p2"];var cols=["var(--green)","var(--blue)"];
        for(var i=0;i<2;i++){
            var pl=state.players[i];
            var ph=pl.properties.length===0?"No properties yet":pl.properties.map(function(pos){return "<span>"+BOARD[pos].name+"</span>";}).join("");
            document.getElementById("p"+(i+1)+"card").innerHTML=
                "<h3><span class='dot "+dots[i]+"'></span><span style='color:"+cols[i]+"'>"+pl.name+"</span></h3>"+
                "<div class='stat'><span class='label'>Cash</span><span class='value'>£"+pl.cash+"</span></div>"+
                "<div class='stat'><span class='label'>Position</span><span class='value'>"+BOARD[pl.position].name+"</span></div>"+
                "<div class='stat'><span class='label'>In Jail?</span><span class='value'>"+(pl.inJail?"Yes ("+pl.jailTurns+"/3)":"No")+"</span></div>"+
                "<div class='props'>"+ph+"</div>";
        }
        document.getElementById("rollBtn").disabled=state.phase!=="roll"||state.gameOver;
        document.getElementById("buyBtn").disabled=state.phase!=="buy"||state.gameOver;
        document.getElementById("endTurnBtn").disabled=(state.phase!=="endturn"&&state.phase!=="buy")||state.gameOver;
    }

    log("Welcome to Text Monopoly! 🎩","system");
    log("Each player starts with £1500.","system");
    log("Roll the dice to begin!","system");
    updateUI();
    </script>
</body>
</html>
"""

filepath = os.path.join(project_folder, "06_jail_and_polish.html")
with open(filepath, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"File saved: {filepath}")
print("This version has the winner banner, improved jail, and scrollbar styling!")

## Section 4 — Edge Cases to Think About

Good software handles unusual situations gracefully. Here are some edge cases in our game:

- **Landing exactly on GO** vs passing over it: both should give £200, but we must not double-count.
- **Chance card sends player to jail** from a position earlier on the board: the "passed GO" check must not trigger incorrectly.
- **Player cannot afford a property**: the buy button should stay disabled.
- **Negative cash**: a player could go negative from rent or tax before the bankrupt check runs.

The current code handles most of these. Identifying and testing edge cases is an important skill in software development.

## Section 5 — Challenge

1. Add a "New Game" button that resets the state to its initial values and clears the console.
2. Make the console display the turn number every 10 turns (e.g., "--- Turn 10 ---").
3. Try modifying the jail rules so that a player can choose to pay £50 immediately to leave jail instead of waiting for doubles.

In **Workshop 07** you will have the complete, final game with all features and some ideas for extensions. Nearly there! 🎉